# 05 — Comparación BoW, TF-IDF, Word2Vec y BETO

Comparación sobre **las mismas reseñas**, con partición común para clasificación, K-Means, Silhouette y t-SNE. Se utilizan los archivos exportados por el equipo.

**Importante:** las métricas se calculan al ejecutar este notebook; no se presuponen resultados. BoW y TF-IDF son representaciones de texto, CBOW/Skip-Gram son variantes de Word2Vec y BETO es un modelo contextual.

## 1. Dependencias y carga de archivos

In [ ]:
!pip -q install pandas numpy scipy scikit-learn matplotlib seaborn tqdm

In [ ]:
from google.colab import files
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz, issparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import Normalizer, StandardScaler

SEMILLA = 42
necesarios = ['bow.npz', 'tfidf.npz', 'ids_resenas.csv', 'vectores_cbow.csv.gz', 'vectores_skip_gram.csv.gz', 'vectores_beto_final.npy', 'metadata_beto_final.csv']
print('Subí estos archivos:', necesarios)
files.upload()

## 2. Lectura y verificación de identificadores
Se detecta automáticamente el archivo CBOW que Colab puede renombrar como `vectores_cbow.csv(1).gz`. Nunca asumimos que el orden de filas coincide sin verificarlo.

In [ ]:
from pathlib import Path

def encontrar_cbow():
    opciones = sorted(Path('.').glob('vectores_cbow*.gz'))
    if not opciones:
        raise FileNotFoundError('Falta el archivo CBOW .gz')
    return opciones[-1]

ids = pd.read_csv('ids_resenas.csv')
meta = pd.read_csv('metadata_beto_final.csv')
cbow = pd.read_csv(encontrar_cbow())
skip = pd.read_csv('vectores_skip_gram.csv.gz')
X_bow_original = load_npz('bow.npz')
X_tfidf_original = load_npz('tfidf.npz')
X_beto_original = np.load('vectores_beto_final.npy', mmap_mode='r')

for nombre, tabla in [('ids',ids),('BETO',meta),('CBOW',cbow),('Skip-Gram',skip)]:
    assert 'id_resena' in tabla, f'{nombre}: falta id_resena'
    assert tabla['id_resena'].notna().all(), f'{nombre}: IDs vacíos'
    assert tabla['id_resena'].is_unique, f'{nombre}: IDs duplicados'
    tabla['id_resena'] = tabla['id_resena'].astype(str)

assert X_bow_original.shape[0] == len(ids) == X_tfidf_original.shape[0]
assert len(meta) == X_beto_original.shape[0]
print('BoW:',X_bow_original.shape,'TF-IDF:',X_tfidf_original.shape)
print('CBOW:',cbow.shape,'Skip-Gram:',skip.shape,'BETO:',X_beto_original.shape)
print('Coincidencia IDs CBOW:', len(set(ids.id_resena) & set(cbow.id_resena)))
print('Coincidencia IDs Skip-Gram:', len(set(ids.id_resena) & set(skip.id_resena)))
print('Coincidencia IDs BETO:', len(set(ids.id_resena) & set(meta.id_resena)))

## 3. Alineación por ID y etiquetas
La matriz BoW y TF-IDF sigue el orden de `ids_resenas.csv`; las otras matrices se reordenan según ese archivo. Solo se incluyen reseñas comunes con polaridad válida.

In [ ]:
# Reindexar sin convertir IDs a números (ej. p1_1, sensacine_...)
ids_base = ids['id_resena'].astype(str)
meta_idx = meta.set_index('id_resena')
cbow_idx = cbow.set_index('id_resena')
skip_idx = skip.set_index('id_resena')
comunes = set(meta_idx.index) & set(cbow_idx.index) & set(skip_idx.index)
mask = ids_base.isin(comunes).to_numpy()
pos_bow = np.flatnonzero(mask)
ids_comunes = ids_base.iloc[pos_bow].tolist()
metadata = meta_idx.loc[ids_comunes].reset_index()

etiqueta = 'polaridad'
assert etiqueta in metadata, 'No existe la columna polaridad en metadata_beto_final.csv'
validos = metadata[etiqueta].notna().to_numpy()
metadata = metadata.loc[validos].reset_index(drop=True)
pos_bow = pos_bow[validos]
ids_finales = metadata['id_resena'].tolist()

cols_cbow = [c for c in cbow_idx.columns if c.startswith('dim_')]
cols_skip = [c for c in skip_idx.columns if c.startswith('dim_')]
assert cols_cbow and cols_skip, 'No se encontraron columnas dim_ en Word2Vec'
X = {
 'BoW': X_bow_original[pos_bow].tocsr(),
 'TF-IDF': X_tfidf_original[pos_bow].tocsr(),
 'Word2Vec CBOW': cbow_idx.loc[ids_finales, cols_cbow].to_numpy(dtype=np.float32),
 'Word2Vec Skip-Gram': skip_idx.loc[ids_finales, cols_skip].to_numpy(dtype=np.float32),
 'BETO': np.asarray(X_beto_original[meta_idx.index.get_indexer(ids_finales)], dtype=np.float32)
}
y = metadata[etiqueta].astype(str).to_numpy()
for nombre, matriz in X.items():
    assert matriz.shape[0] == len(y), f'Desalineación en {nombre}'
    assert np.isfinite(matriz.data if issparse(matriz) else matriz).all(), f'Valores no finitos en {nombre}'
print('Reseñas comunes con etiqueta:', len(y))
print('Clases:',pd.Series(y).value_counts().to_dict())
print({k:v.shape for k,v in X.items()})

## 4. Clasificación supervisada (misma partición)
Clasificamos **polaridad** con regresión logística, partición estratificada 80/20 y semilla fija. Para evitar fuga de información, el clasificador se ajusta solo con entrenamiento. Las representaciones BoW/TF-IDF/Word2Vec/BETO fueron previamente calculadas sobre el corpus; esto es una comparación de representaciones, no una evaluación de entrenamiento de embeddings desde cero.

In [ ]:
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=SEMILLA, stratify=y)
resultados_clasificacion = []
reportes = {}
for nombre, matriz in X.items():
    modelo_clf = LogisticRegression(max_iter=1500, class_weight='balanced', solver='lbfgs')
    modelo_clf.fit(matriz[train_idx], y[train_idx])
    pred = modelo_clf.predict(matriz[test_idx])
    resultados_clasificacion.append({
        'representacion':nombre,
        'accuracy':accuracy_score(y[test_idx], pred),
        'f1_macro':f1_score(y[test_idx], pred, average='macro', zero_division=0),
        'f1_weighted':f1_score(y[test_idx], pred, average='weighted', zero_division=0)
    })
    reportes[nombre] = classification_report(y[test_idx], pred, zero_division=0)
clasificacion_df = pd.DataFrame(resultados_clasificacion).sort_values('f1_macro', ascending=False)
display(clasificacion_df)
clasificacion_df.to_csv('comparacion_clasificacion.csv', index=False, encoding='utf-8-sig')
print('Reporte BETO:\n',reportes['BETO'])

## 5. Clustering K-Means y Silhouette
Silhouette se calcula sobre una **muestra común** para reducir el costo y permitir comparabilidad. Para BoW y TF-IDF reducimos dimensiones con TruncatedSVD antes de K-Means; para las representaciones densas se usa normalización L2. La reducción SVD se realiza solo con fines de análisis exploratorio.

In [ ]:
n_muestra = min(1200, len(y))
# Muestra estratificada, con tamaño y semilla idénticos para todas las representaciones.
_, muestra_idx = train_test_split(np.arange(len(y)), test_size=n_muestra, random_state=SEMILLA, stratify=y)

def preparar_clustering(matriz, nombre):
    Z = matriz[muestra_idx]
    if issparse(Z):
        ncomp = min(100, Z.shape[0]-1, Z.shape[1]-1)
        Z = TruncatedSVD(n_components=ncomp, random_state=SEMILLA).fit_transform(Z)
    else:
        Z = np.asarray(Z, dtype=np.float32)
    return Normalizer().fit_transform(Z)

Z_cluster = {nombre: preparar_clustering(matriz, nombre) for nombre,matriz in X.items()}
k = len(np.unique(y))
resultados_cluster = []
clusters = {}
for nombre, Z in Z_cluster.items():
    km = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10)
    etiquetas_cluster = km.fit_predict(Z)
    clusters[nombre] = etiquetas_cluster
    sil = silhouette_score(Z, etiquetas_cluster, metric='cosine', sample_size=min(1000,len(Z)), random_state=SEMILLA) if len(np.unique(etiquetas_cluster)) > 1 else np.nan
    resultados_cluster.append({'representacion':nombre, 'k':k, 'silhouette_coseno':sil, 'ARI_vs_polaridad':adjusted_rand_score(y[muestra_idx], etiquetas_cluster)})
cluster_df = pd.DataFrame(resultados_cluster).sort_values('silhouette_coseno', ascending=False)
display(cluster_df)
cluster_df.to_csv('comparacion_clustering.csv', index=False, encoding='utf-8-sig')

## 6. Visualización t-SNE
Se usa **la misma muestra de reseñas** para cada representación. t-SNE es una proyección exploratoria y sus ejes no tienen significado directo; no debe utilizarse para comparar distancias absolutas entre gráficos.

In [ ]:
n_tsne = min(600, len(muestra_idx))
sub = np.arange(n_tsne)
fig, ejes = plt.subplots(2, 3, figsize=(18, 11))
ejes = ejes.ravel()
clases = sorted(np.unique(y[muestra_idx][sub]))
mapa_colores = {clase:i for i,clase in enumerate(clases)}
colores = [mapa_colores[v] for v in y[muestra_idx][sub]]
for ax, (nombre, Z) in zip(ejes, Z_cluster.items()):
    coords = TSNE(n_components=2, perplexity=min(30, max(5,(n_tsne-1)//3)), init='pca', learning_rate='auto', random_state=SEMILLA).fit_transform(Z[sub])
    ax.scatter(coords[:,0], coords[:,1], c=colores, cmap='tab10', s=10, alpha=.75)
    ax.set_title(nombre)
    ax.set_xticks([]);ax.set_yticks([])
for ax in ejes[len(Z_cluster):]: ax.axis('off')
fig.suptitle('t-SNE por representación — colores: polaridad', fontsize=16)
fig.tight_layout()
fig.savefig('tsne_comparacion.png', dpi=160, bbox_inches='tight')
plt.show()
print('Clases y códigos de color:', mapa_colores)

## 7. Resumen y exportación
Se exportan tablas y figura para incorporarlas al dashboard en Plotly Dash. Las conclusiones se redactan **después de observar las métricas**.

In [ ]:
resumen = clasificacion_df.merge(cluster_df, on='representacion', how='left')
display(resumen)
resumen.to_csv('resumen_comparacion.csv', index=False, encoding='utf-8-sig')
metadata[['id_resena', 'polaridad']].to_csv('ids_comparacion.csv', index=False, encoding='utf-8-sig')

import zipfile
salidas = ['comparacion_clasificacion.csv','comparacion_clustering.csv','resumen_comparacion.csv','tsne_comparacion.png','ids_comparacion.csv']
with zipfile.ZipFile('RESULTADOS_COMPARACION.zip','w',zipfile.ZIP_DEFLATED) as z:
    for archivo in salidas: z.write(archivo)
files.download('RESULTADOS_COMPARACION.zip')
print('Comparación finalizada. Revisá las tablas y gráficos antes de redactar conclusiones.')